# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện
**Sinh viên thực hiện:** Nguyễn Văn Ước  
**MSSV:** 2A202602445  
**Học phần:** VinUniversity AICB 2026 — Track 4: Mạng Nơ-ron và Huấn Luyện  

---
### Mục tiêu của bài thực hành:
1. **Part 0:** Phân chia tập dữ liệu `train`/`eval` theo metadata, phân tầng validation 20% từ train, chuẩn hoá 10 đặc trưng số liên tục (chỉ dùng thống kê trên train).
2. **Part 1:** Định nghĩa kiến trúc `MLP` (`M-base` 47 879 tham số), kiểm tra shape, kiểm tra "sức khoẻ" ban đầu (loss bước 0 $\approx \ln 7 \approx 1.946$, quá khớp 20 mẫu, gradient flow).
3. **Part 2:** Pipeline huấn luyện chuẩn mực `run_experiment(cfg, data)`, chạy baseline `M-base` trên 3 seed để đo độ nhiễu và ngưỡng $2\sigma$.
4. **Part 3:** Thực hiện các bộ thí nghiệm phủ đủ **7 chủ đề** theo GUIDE/RUBRIC:
   - Chủ đề 1: Hàm mất mát (Cross-Entropy vs MSE).
   - Chủ đề 2: Bộ tối ưu hoá (SGD, SGD+Momentum, Adam, AdamW với khảo sát learning rate).
   - Chủ đề 3: Hyper-parameters (Batch size 128, 512, 2048; Kiến trúc M-wide, M-deep).
   - Chủ đề 4: Dropout ($q = 0.1, 0.3$).
   - Chủ đề 5: Gradient clipping (Khảo sát ở lr thông thường và phản chứng ở lr cao).
   - Chủ đề 6: Mixed precision (FP32 vs FP16 với GradScaler).
   - Chủ đề 7: Khởi tạo tham số (`he`, `xavier`, `normal`, `zeros` và thống kê kích hoạt).
5. **Part 4:** Đánh giá cấu hình tốt nhất trên tập `eval` bằng `scripts/evaluate.py`, phân tích lỗi chi tiết từng lớp, xuất bảng `experiments.xlsx` và các biểu đồ phục vụ báo cáo `REPORT.md`.


In [ ]:
# ===== Thiết lập môi trường và cấu hình đường dẫn =====
import os
import sys
import json
import time
import subprocess
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# 1. Xác định môi trường thực thi (Google Colab hoặc Local)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Môi trường: Google Colab")
    # Luôn cập nhật mã mới nhất từ repo GitHub của bạn
    if os.path.exists("/content/K4-Track4-Day1-Neuralnetwork"):
        %cd /content/K4-Track4-Day1-Neuralnetwork
        !git remote set-url origin https://github.com/nvuoc/K4-Track4-Day1-Neuralnetwork.git
        !git fetch origin main
        !git reset --hard origin/main
    elif not os.path.exists("data"):
        %cd /content
        !git clone https://github.com/nvuoc/K4-Track4-Day1-Neuralnetwork.git
        %cd /content/K4-Track4-Day1-Neuralnetwork
    REPO_ROOT = "."
    OUT_DIR = "submission_2A202602445"
else:
    print("Môi trường: Local / Jupyter")
    if os.path.exists("data"):
        REPO_ROOT = "."
        OUT_DIR = "submission_2A202602445"
    elif os.path.exists("../data"):
        REPO_ROOT = ".."
        OUT_DIR = "../submission_2A202602445"
    elif os.path.exists("../../data"):
        REPO_ROOT = "../.."
        OUT_DIR = ".."
    else:
        REPO_ROOT = "."
        OUT_DIR = "submission_2A202602445"

# Thêm thư mục code vào sys.path để import các module
code_dir = os.path.abspath(os.path.join(REPO_ROOT, "code"))
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

# 2. Thiết bị tính toán
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Thiết bị tính toán: {device}")
if device == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

# 3. Tạo thư mục output nếu chưa có
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "results"), exist_ok=True)
print(f"Thư mục gốc Repo:   {os.path.abspath(REPO_ROOT)}")
print(f"Thư mục nộp bài:    {os.path.abspath(OUT_DIR)}")

# Import và tự động reload các module
import importlib
import data, model, optimizer, train, plots, results_table
importlib.reload(data)
importlib.reload(model)
importlib.reload(optimizer)
importlib.reload(train)
importlib.reload(plots)
importlib.reload(results_table)

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, write_predictions
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
print("Đã import toàn bộ modules thành công!")


## Part 0 — Chuẩn bị Dữ liệu
Dữ liệu Forest CoverType gồm 581 012 mẫu, 54 đặc trưng (10 đặc trưng số liên tục, 44 cột nhị phân one-hot) và 7 lớp rừng.
- Dữ liệu được chia theo `split_metadata.csv` (cố định) thành `train` (464 809 mẫu) và `eval` (116 203 mẫu).
- Tách `validation` 20% từ `train` (phân tầng theo nhãn, seed 42) $\rightarrow$ 371 847 mẫu train, 92 962 mẫu val.
- Chuẩn hoá 10 đặc trưng số liên tục **chỉ dùng thống kê mean/std của phần train còn lại**, áp dụng cho val và eval. Tuyệt đối không rò rỉ thông tin từ eval.


In [ ]:
# Đảm bảo dữ liệu đã được chia thành train.npz và eval.npz
split_script = os.path.join(REPO_ROOT, "scripts", "split_data.py")
proc_dir = os.path.join(REPO_ROOT, "data", "processed")
train_npz = os.path.join(proc_dir, "train.npz")
eval_npz = os.path.join(proc_dir, "eval.npz")

if not os.path.exists(train_npz) or not os.path.exists(eval_npz):
    print("Chưa thấy train.npz / eval.npz, tiến hành chạy split_data.py...")
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    subprocess.run([sys.executable, split_script], cwd=REPO_ROOT, env=env, check=True)

# Tải và chuẩn bị dữ liệu, chuyển tensor lên thiết bị tính toán
print("Đang nạp và chuẩn hóa dữ liệu...")
t0_data = time.time()
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=proc_dir)
print(f"Dữ liệu đã sẵn sàng trong {time.time() - t0_data:.2f} giây.")

# Kiểm tra mean và std của 10 đặc trưng số liên tục trên tập train
mean_num = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
std_num = data["X_tr"][:, :10].std(dim=0).cpu().numpy()
print("\nKiểm tra chuẩn hoá 10 đặc trưng số trên X_tr:")
print("  Mean:", np.round(mean_num, 4))
print("  Std: ", np.round(std_num, 4))
assert np.allclose(mean_num, 0.0, atol=1e-2), "Lỗi: Mean tập train khác 0!"
assert np.allclose(std_num, 1.0, atol=1e-2), "Lỗi: Std tập train khác 1!"
print("=> Hoàn tất kiểm tra Part 0: Không có hiện tượng rò rỉ dữ liệu (data leakage)!")


**Nhận xét Part 0:**
- Tập train sau tách val có 371 847 mẫu; tập val có 92 962 mẫu; tập eval có 116 203 mẫu.
- Chiến lược ngây thơ "luôn đoán lớp đa số" (lớp 1) đạt accuracy là **0.4876** trên tập val (tương ứng macro-F1 chỉ $\approx 0.094$). Đây là mốc cơ sở tối thiểu mà bất kỳ mô hình học máy nào cũng phải vượt qua.
- Thống kê chuẩn hoá (mean và std) chỉ tính trên phần train còn lại sau khi tách val nhằm đảm bảo tính toàn vẹn của mô hình, không cho mô hình "nhìn trước" phân phối của val hay eval.


## Part 1 — Định nghĩa Model và Kiểm tra "Sức Khoẻ" Ban Đầu
Mô hình `MLP` theo chuẩn:
- `M-base`: $54 \rightarrow 256 \rightarrow 128 \rightarrow 7$, ReLU tại mọi lớp ẩn, có bias ở tất cả các lớp linear.
- Không BatchNorm, không residual connection. Dropout (nếu có) chỉ đặt sau ReLU của lớp ẩn.
- Lớp cuối cùng trả về **logits thô** $(B, 7)$, tuyệt đối không softmax trong model.
- Tổng số tham số bắt buộc: **47 879 tham số**.


In [ ]:
set_seed(42)

# 1. Khởi tạo model M-base và kiểm tra số tham số
model_test = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model_test)
print(f"1. Tổng số tham số của M-base: {n_params} | Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]}")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Lỗi số tham số: {n_params} != 47879"

# 2. Kiểm tra shape với batch ngẫu nhiên (8, 54)
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model_test(dummy_x)
print(f"2. Logits output shape: {dummy_logits.shape} | Kỳ vọng: (8, 7)")
assert dummy_logits.shape == (8, 7), "Lỗi shape đầu ra!"

# 3. Phép thử 1: Loss bước 0 trên tập val
step0_eval = evaluate(model_test, data["X_val"], data["y_val"], loss_name="ce")
step0_loss = step0_eval["loss"]
ln7 = float(np.log(7))
print(f"3. Loss bước 0 trên val: {step0_loss:.4f} (so với ln(7) = {ln7:.4f}, chênh lệch = {abs(step0_loss - ln7):.4f})")

# 4. Phép thử 2: Quá khớp trên 20 mẫu (Overfitting 20 samples)
print("4. Chạy phép thử quá khớp 20 mẫu...")
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt_overfit = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
x_20 = data["X_tr"][:20]
y_20 = data["y_tr"][:20]

overfit_loss_history = []
for step in range(250):
    overfit_model.train()
    opt_overfit.zero_grad()
    logits_20 = overfit_model(x_20)
    loss_20 = torch.nn.functional.cross_entropy(logits_20, y_20)
    loss_20.backward()
    opt_overfit.step()
    overfit_loss_history.append(loss_20.item())

overfit_model.eval()
with torch.no_grad():
    acc_20 = (overfit_model(x_20).argmax(dim=-1) == y_20).float().mean().item()
print(f"   => Loss cuối: {overfit_loss_history[-1]:.6f} | Accuracy 20 mẫu: {acc_20*100:.1f}%")
assert acc_20 == 1.0, "Model không thể nhớ hoàn hảo 20 mẫu!"

plt.figure(figsize=(6, 3))
plt.plot(overfit_loss_history, color="#1f77b4", lw=2)
plt.title("Phép thử sức khoẻ: Quá khớp 20 mẫu (Loss -> 0)")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figures", "health_check_overfit20.png"), dpi=150)
plt.show()

# 5. Phép thử 3: Gradient có chảy tới mọi tham số không?
print("5. Kiểm tra gradient của từng tham số:")
test_model_grad = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
out = test_model_grad(data["X_tr"][:32])
loss = torch.nn.functional.cross_entropy(out, data["y_tr"][:32])
loss.backward()

for name, param in test_model_grad.named_parameters():
    assert param.grad is not None, f"Gradient của {name} bị None!"
    gnorm = param.grad.norm().item()
    assert gnorm > 0, f"Gradient của {name} bằng 0!"
    print(f"   {name:25s}: shape {str(tuple(param.shape)):15s} | grad_norm = {gnorm:.6f}")
print("=> Tất cả tham số đều nhận được gradient hợp lệ!")


**Nhận xét Part 1:**
- Kiến trúc `M-base` được khởi tạo chuẩn xác với đúng **47 879 tham số** (54×256 + 256 + 256×128 + 128 + 128×7 + 7 = 47 879).
- Loss bước 0 xấp xỉ $\ln 7 \approx 1.946$, chứng tỏ trọng số ban đầu không bị thiên lệch về một lớp nào và không bị bão hoà.
- Mô hình quá khớp 100% trên 20 mẫu chỉ sau khoảng 100 bước huấn luyện, loss giảm về gần 0. Điều này xác nhận vòng lặp huấn luyện, việc tính loss, zero_grad, và cập nhật trọng số hoạt động chính xác.
- Gradient chảy đều qua toàn bộ các lớp (W1, b1, W2, b2, W3, b3), không bị vanishing hay dead neurons ngay từ đầu.


## Part 2 — Pipeline Huấn luyện, Baseline và Độ Nhiễu Giữa Các Seed
- Baseline bắt buộc: M-base, khởi tạo He, hàm mất mát Cross-Entropy, bộ tối ưu SGD+momentum 0.9, batch size 512, 20 epochs.
- Để thiết lập cơ sở so sánh khoa học ("A tốt hơn B"), ta chạy Baseline trên **3 seed khác nhau** (`base-s1`, `base-s2`, `base-s3`).
- Đo giá trị trung bình $\pm$ độ lệch chuẩn $\sigma$, xác định **ngưỡng nhiễu $2\sigma$**. Mọi khác biệt nhỏ hơn $2\sigma$ sẽ được coi là chưa đủ ý nghĩa thống kê.


In [ ]:
# Chạy Baseline M-base với 3 seed
baseline_results = []
baseline_seeds = [1, 2, 3]

print("=== BẮT ĐẦU HUẤN LUYỆN BASELINE VỚI 3 SEED ===")
for s in baseline_seeds:
    cfg_base = {
        **DEFAULT_CFG,
        "exp_id": f"base-s{s}",
        "group": "baseline",
        "description": f"Baseline M-base (seed {s})",
        "optimizer": "sgd_momentum",
        "lr": 0.05,
        "weight_decay": 0.0,
        "momentum": 0.9,
        "batch": 512,
        "epochs": 20,
        "hidden": (256, 128),
        "dropout": 0.0,
        "init": "he",
        "seed": s,
    }
    print(f"\n--- Đang chạy {cfg_base['exp_id']} ---")
    res = run_experiment(cfg_base, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg_base['exp_id']}.png"))
    baseline_results.append(res)
    sm = res["summary"]
    print(f"Xong {cfg_base['exp_id']}: Best Val Loss: {sm['best_val_loss']:.4f} (ep {sm['best_epoch']}) | Val Acc: {sm['val_acc']:.4f} | Val Macro-F1: {sm['val_macro_f1']:.4f}")

# Tính toán độ nhiễu thống kê
val_accs = [r["summary"]["val_acc"] for r in baseline_results]
val_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
val_losses = [r["summary"]["best_val_loss"] for r in baseline_results]

mean_acc, std_acc = float(np.mean(val_accs)), float(np.std(val_accs, ddof=1))
mean_f1, std_f1 = float(np.mean(val_f1s)), float(np.std(val_f1s, ddof=1))
mean_loss, std_loss = float(np.mean(val_losses)), float(np.std(val_losses, ddof=1))
threshold_2sigma = 2 * std_f1

print("\n" + "="*60)
print(f"KẾT QUẢ BASELINE QUA {len(baseline_seeds)} SEED:")
print(f"  Val Accuracy:      {mean_acc:.4f} ± {std_acc:.4f}")
print(f"  Val Macro-F1:      {mean_f1:.4f} ± {std_f1:.4f}")
print(f"  Best Val Loss:     {mean_loss:.4f} ± {std_loss:.4f}")
print(f"  NGƯỠNG NHIỄU 2σ (Val Macro-F1): {threshold_2sigma:.4f}")
print("="*60)


**Nhận xét Part 2:**
- Baseline huấn luyện ổn định qua 20 epoch. Đường loss giảm đều và không có hiện tượng divergence.
- Val accuracy đạt $\approx 0.82$, vượt xa mốc đoán đa số 0.4876; Val macro-F1 đạt $\approx 0.77 - 0.79$.
- Độ lệch chuẩn $\sigma$ giữa các seed cung cấp ngưỡng nhiễu $2\sigma$. Ở các phần sau, một cải tiến chỉ được coi là có ý nghĩa thực sự nếu mức tăng macro-F1 lớn hơn $2\sigma$.


## Part 3 — Thí nghiệm Chuyên sâu (Phủ đủ 7 Chủ đề)

### Chủ đề 1 — Hàm mất mát (`loss`): Cross-Entropy vs MSE
- **Dự đoán trước khi chạy:** Cross-Entropy (CE) sẽ hội tụ nhanh hơn và cho F1 cao hơn MSE. Lý do: đạo hàm của hàm mất mát CE kết hợp với softmax cho gradient tỷ lệ thuận với độ lệch $(p - y)$, không bị bão hoà ngay cả khi mô hình dự đoán sai lệch nhiều; ngược lại MSE tính trên logit hoặc bị triệt tiêu gradient ở vùng xác suất bão hòa.
- **Lưu ý:** Không so sánh trực tiếp giá trị loss giữa CE và MSE vì khác thang đo; so sánh bằng accuracy, macro-F1 và tốc độ hội tụ.


In [ ]:
# Thí nghiệm Chủ đề 1: MSE Loss
cfg_mse = {
    **DEFAULT_CFG,
    "exp_id": "loss-mse",
    "group": "loss",
    "description": "MSE Loss trên one-hot",
    "loss": "mse",
    "optimizer": "sgd_momentum",
    "lr": 0.05,
    "seed": 1,
    "epochs": 20,
}
print(f"Đang chạy thí nghiệm {cfg_mse['exp_id']}...")
res_mse = run_experiment(cfg_mse, data)
save_result(res_mse, os.path.join(OUT_DIR, "results"))
plot_run(res_mse, os.path.join(OUT_DIR, "figures", f"{cfg_mse['exp_id']}.png"))

# So sánh CE vs MSE
plot_compare([baseline_results[0], res_mse], "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_loss.png"), 
             title="So sánh Hàm mất mát: CE (base-s1) vs MSE (loss-mse)")

delta_f1 = res_mse["summary"]["val_macro_f1"] - baseline_results[0]["summary"]["val_macro_f1"]
print(f"MSE Val Macro-F1: {res_mse['summary']['val_macro_f1']:.4f} vs CE: {baseline_results[0]['summary']['val_macro_f1']:.4f} (Chênh lệch: {delta_f1:+.4f})")


### Chủ đề 2 — Bộ tối ưu hoá (`optimizer`)
Khảo sát 4 bộ tối ưu:
- **SGD** (thuần): không có quán tính, dễ bị kẹt ở hẻm núi gradient hẹp.
- **SGD + Momentum (0.9)**: tích luỹ vận tốc $v \leftarrow \mu v + g$, vượt qua điểm uốn và dao động dốc.
- **Adam**: điều chỉnh tốc độ học riêng cho từng tham số dựa trên mô-men bậc 1 $m$ và bậc 2 $v$.
- **AdamW**: tách rời weight decay khỏi gradient update ($w \leftarrow w - \eta \lambda w - \eta \hat{m}/(\sqrt{\hat{v}} + \epsilon)$).
- **Dự đoán:** Adam/AdamW với $lr = 0.001$ sẽ hội tụ nhanh hơn nhiều so với SGD và cho macro-F1 cao nhất.


In [ ]:
# Thí nghiệm Chủ đề 2: Bộ tối ưu hoá
opt_configs = [
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.05", "group": "optimizer", "description": "SGD lr=0.05", "optimizer": "sgd", "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.2", "group": "optimizer", "description": "SGD lr=0.2", "optimizer": "sgd", "lr": 0.2, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr1e-3", "group": "optimizer", "description": "Adam lr=0.001", "optimizer": "adam", "lr": 0.001, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr3e-4", "group": "optimizer", "description": "Adam lr=0.0003", "optimizer": "adam", "lr": 0.0003, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "description": "AdamW lr=0.001 wd=0.01", "optimizer": "adamw", "lr": 0.001, "weight_decay": 0.01, "seed": 1},
]

opt_results = [baseline_results[0]]  # Kèm baseline (SGD+momentum)
for cfg in opt_configs:
    print(f"Đang chạy thí nghiệm {cfg['exp_id']}...")
    res = run_experiment(cfg, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg['exp_id']}.png"))
    opt_results.append(res)
    print(f"  Xong {cfg['exp_id']} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f} | Val Acc: {res['summary']['val_acc']:.4f}")

# Vẽ đồ thị so sánh nhóm optimizer
plot_compare(opt_results, "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_optimizer.png"), 
             title="So sánh các Bộ Tối Ưu Hoá (Val Macro-F1)")


### Chủ đề 3 — Hyper-parameters (`hparam`): Batch Size, Độ rộng (`M-wide`), Độ sâu (`M-deep`)
- **Batch Size:** Thử nghiệm 128, 512 (baseline) và 2048.
  - *Dự đoán:* Batch 128 thực hiện nhiều bước cập nhật hơn trong 1 epoch nên học nhanh hơn ở các epoch đầu; Batch 2048 chạy rất nhanh về thời gian tính toán nhưng ít bước cập nhật hơn.
- **Kiến trúc mô hình:**
  - `M-wide`: $54 \rightarrow 512 \rightarrow 256 \rightarrow 7$ (161 287 tham số).
  - `M-deep`: $54 \rightarrow 256 \rightarrow 128 \rightarrow 64 \rightarrow 7$ (55 687 tham số).
  - *Dự đoán:* `M-wide` tăng dung lượng mô hình đáng kể, giúp biểu diễn tốt hơn mối quan hệ phi tuyến phức tạp của các loại đất và độ cao địa hình, nâng cao macro-F1.


In [ ]:
# Thí nghiệm Chủ đề 3: Hyper-parameters
hparam_configs = [
    {**DEFAULT_CFG, "exp_id": "hp-batch-128", "group": "hparam", "description": "Batch 128 lr=0.02", "batch": 128, "lr": 0.02, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hp-batch-2048", "group": "hparam", "description": "Batch 2048 lr=0.1", "batch": 2048, "lr": 0.1, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hp-wide", "group": "hparam", "description": "M-wide (512-256)", "hidden": (512, 256), "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "hp-deep", "group": "hparam", "description": "M-deep (256-128-64)", "hidden": (256, 128, 64), "lr": 0.05, "seed": 1},
]

hparam_results = [baseline_results[0]]
for cfg in hparam_configs:
    print(f"Đang chạy thí nghiệm {cfg['exp_id']}...")
    res = run_experiment(cfg, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg['exp_id']}.png"))
    hparam_results.append(res)
    print(f"  Xong {cfg['exp_id']} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f} | Thời gian/epoch: {res['summary']['time_per_epoch_s']:.2f}s")

plot_compare(hparam_results, "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_hparam.png"), 
             title="So sánh Hyper-parameters (Batch size, M-wide, M-deep)")


### Chủ đề 4 — Dropout (`dropout`): $q = 0.1$ và $q = 0.3$
- **Quy tắc:** Dropout chỉ đặt **sau ReLU của các lớp ẩn**.
- **Dự đoán trước khi chạy:** Do tập dữ liệu train rất lớn (371 847 mẫu) và mô hình `M-base` có 47 879 tham số (tỷ lệ mẫu/tham số $\approx 7.7$), mô hình chưa bị quá khớp (overfitting) nghiêm trọng ở 20 epoch. Vì vậy, dropout cao ($q=0.3$) có thể làm chậm quá trình học và làm giảm nhẹ điểm số trên val.


In [ ]:
# Thí nghiệm Chủ đề 4: Dropout
dropout_configs = [
    {**DEFAULT_CFG, "exp_id": "drop-0.1", "group": "dropout", "description": "Dropout q=0.1", "dropout": 0.1, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "drop-0.3", "group": "dropout", "description": "Dropout q=0.3", "dropout": 0.3, "seed": 1},
]

dropout_results = [baseline_results[0]]
for cfg in dropout_configs:
    print(f"Đang chạy thí nghiệm {cfg['exp_id']}...")
    res = run_experiment(cfg, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg['exp_id']}.png"))
    dropout_results.append(res)
    print(f"  Xong {cfg['exp_id']} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f}")

plot_compare(dropout_results, "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_dropout.png"), 
             title="So sánh Dropout (q=0.0 vs q=0.1 vs q=0.3)")


### Chủ đề 5 — Cắt gradient (`clipping`)
- Công thức: $g \leftarrow g \cdot \min(1, c / \|g\|)$ với chuẩn $L_2$ toàn cục đo **trước khi cắt**.
- **Dự đoán:**
  - Ở tốc độ học thông thường ($lr = 0.05$), gradient norm của baseline dao động trong khoảng $0.5 - 2.0$. Việc cắt với $c = 1.0$ sẽ ít gây ảnh hưởng lớn.
  - **Thí nghiệm phản chứng:** Khi tăng $lr$ lên rất cao ($lr = 0.5$), nếu không cắt gradient, mô hình sẽ bị dao động mạnh hoặc phân kỳ (diverged / loss cao); khi bật clip $c = 1.0$, gradient clipping sẽ cứu vãn quá trình huấn luyện và giữ loss ổn định.


In [ ]:
# Thí nghiệm Chủ đề 5: Gradient Clipping
clip_configs = [
    {**DEFAULT_CFG, "exp_id": "clip-c1.0", "group": "clipping", "description": "Clip c=1.0 ở lr=0.05", "clip_norm": 1.0, "lr": 0.05, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "clip-highlr-noclip", "group": "clipping", "description": "High lr=0.5 không clip", "clip_norm": None, "lr": 0.5, "seed": 1},
    {**DEFAULT_CFG, "exp_id": "clip-highlr-clip1.0", "group": "clipping", "description": "High lr=0.5 có clip c=1.0", "clip_norm": 1.0, "lr": 0.5, "seed": 1},
]

clip_results = [baseline_results[0]]
for cfg in clip_configs:
    print(f"Đang chạy thí nghiệm {cfg['exp_id']}...")
    res = run_experiment(cfg, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg['exp_id']}.png"))
    clip_results.append(res)
    print(f"  Xong {cfg['exp_id']} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f} | Diverged: {res['summary']['diverged']}")

plot_compare(clip_results, "val_loss", 
             os.path.join(OUT_DIR, "figures", "compare_clipping.png"), 
             title="So sánh Gradient Clipping (Val Loss)")


### Chủ đề 6 — Mixed Precision (`amp`): FP32 vs FP16
- Thử nghiệm độ chính xác nửa (FP16) kết hợp `torch.amp.GradScaler`.
- **Dự đoán:**
  - Trên GPU kiến trúc Tensor Cores (như T4 hoặc RTX), FP16 giúp giảm mức tiêu thụ bộ nhớ VRAM.
  - Tuy nhiên với mạng MLP kích thước nhỏ (47k tham số), chi phí gọi kernel và overhead của GradScaler có thể khiến thời gian huấn luyện tương đương hoặc chỉ nhanh hơn một chút so với FP32.


In [ ]:
# Thí nghiệm Chủ đề 6: Mixed Precision
cfg_amp = {
    **DEFAULT_CFG,
    "exp_id": "amp-fp16",
    "group": "amp",
    "description": "FP16 Mixed Precision",
    "precision": "fp16",
    "seed": 1,
}

print(f"Đang chạy thí nghiệm {cfg_amp['exp_id']}...")
res_amp = run_experiment(cfg_amp, data)
save_result(res_amp, os.path.join(OUT_DIR, "results"))
plot_run(res_amp, os.path.join(OUT_DIR, "figures", f"{cfg_amp['exp_id']}.png"))

plot_compare([baseline_results[0], res_amp], "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_amp.png"), 
             title="So sánh Precision: FP32 vs FP16")

print(f"FP32: {baseline_results[0]['summary']['time_per_epoch_s']:.2f}s/epoch | VRAM: {baseline_results[0]['summary']['peak_mem_MB']:.1f}MB | F1: {baseline_results[0]['summary']['val_macro_f1']:.4f}")
print(f"FP16: {res_amp['summary']['time_per_epoch_s']:.2f}s/epoch | VRAM: {res_amp['summary']['peak_mem_MB']:.1f}MB | F1: {res_amp['summary']['val_macro_f1']:.4f}")


### Chủ đề 7 — Khởi tạo Tham số (`init`): He vs Xavier vs Normal vs Zeros
Khảo sát các phương pháp khởi tạo:
- `he` (Kaiming Normal): phương sai $\text{Var}[W] = 2 / n_{\text{in}}$, phù hợp cho hàm kích hoạt ReLU.
- `xavier` (Xavier Normal): $\text{Var}[W] = 2 / (n_{\text{in}} + n_{\text{out}})$.
- `normal`: $W \sim \mathcal{N}(0, 0.01^2)$ (độ lệch chuẩn quá nhỏ dẫn đến tín hiệu suy kiệt).
- `zeros`: $W = 0$.
- **Dự đoán:**
  - `zeros` sẽ **hoàn toàn thất bại** (không học được): khi mọi trọng số bằng 0, mọi nơ-ron trong cùng một lớp ẩn đều nhận tín hiệu vào và gradient giống hệt nhau (mất tính phá vỡ đối xứng - symmetry breaking), khiến mô hình chỉ tương đương 1 nơ-ron đơn lẻ.
  - `he` sẽ cho phân phối kích hoạt ổn định nhất qua các lớp sâu.


In [ ]:
# Thí nghiệm Chủ đề 7: Khởi tạo tham số
init_configs = [
    {**DEFAULT_CFG, "exp_id": "init-zeros", "group": "init", "description": "Zeros init W=0", "init": "zeros", "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-normal", "group": "init", "description": "Normal N(0, 0.01^2)", "init": "normal", "seed": 1},
    {**DEFAULT_CFG, "exp_id": "init-xavier", "group": "init", "description": "Xavier Normal", "init": "xavier", "seed": 1},
]

init_results = [baseline_results[0]]  # base-s1 dùng He init
print("Đo độ lệch chuẩn kích hoạt (std) ở bước 0 với một lô dữ liệu val:")
batch_val_sample = data["X_val"][:512]

for init_name in ["he", "xavier", "normal", "zeros"]:
    m_temp = MLP(hidden=(256, 128), dropout=0.0, init=init_name).to(device)
    stds = activation_stats(m_temp, batch_val_sample)
    print(f"  Khởi tạo {init_name:7s} -> std sau các lớp linear: {[round(s, 5) for s in stds]}")

for cfg in init_configs:
    print(f"\nĐang chạy thí nghiệm {cfg['exp_id']}...")
    res = run_experiment(cfg, data)
    save_result(res, os.path.join(OUT_DIR, "results"))
    plot_run(res, os.path.join(OUT_DIR, "figures", f"{cfg['exp_id']}.png"))
    init_results.append(res)
    print(f"  Xong {cfg['exp_id']} | Step 0 Loss: {res['summary']['step0_loss']:.4f} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f}")

plot_compare(init_results, "val_macro_f1", 
             os.path.join(OUT_DIR, "figures", "compare_init.png"), 
             title="So sánh Khởi tạo Trọng số: He vs Xavier vs Normal vs Zeros")


## Part 4 — Đánh giá Cuối cùng trên Tập Eval, Lập Bảng và Báo cáo
**Quy trình chuẩn mực:**
1. Chọn cấu hình tốt nhất **HOÀN TOÀN DỰA TRÊN TẬP VALIDATION** (không nhìn trước tập eval).
2. Huấn luyện cấu hình cuối cùng (`final-best`) và nạp trọng số ở `best_epoch` (dựa trên val loss thấp nhất).
3. Dự đoán nhãn trên toàn bộ 116 203 mẫu của tập eval ở chế độ `eval()`.
4. Ghi file `predictions_eval.csv` theo định dạng `row_id,pred` (0..6).
5. Thực thi script chấm điểm chính thức `scripts/evaluate.py` để xuất file `eval_result.json`.
6. Phân tích chi tiết ma trận nhầm lẫn và F1 từng lớp.
7. Xuất toàn bộ kết quả vào `experiments.xlsx`.


In [ ]:
# 1. Huấn luyện Cấu hình Cuối cùng (Final Model)
# Dựa trên kết quả val: Bộ tối ưu Adam với lr=0.001 hoặc M-wide cho kết quả vượt trội
cfg_final = {
    **DEFAULT_CFG,
    "exp_id": "final-best",
    "group": "final",
    "description": "Cấu hình cuối: M-wide, Adam lr=0.001",
    "optimizer": "adam",
    "lr": 0.001,
    "hidden": (512, 256),
    "batch": 512,
    "epochs": 20,
    "seed": 1,
}

print("=== HUẤN LUYỆN CẤU HÌNH CUỐI CÙNG ===")
res_final = run_experiment(cfg_final, data)
save_result(res_final, os.path.join(OUT_DIR, "results"))
plot_run(res_final, os.path.join(OUT_DIR, "figures", f"{cfg_final['exp_id']}.png"))
print(f"Cấu hình cuối cùng trên Val: Accuracy = {res_final['summary']['val_acc']:.4f} | Macro-F1 = {res_final['summary']['val_macro_f1']:.4f}")

# 2. Dự đoán trên tập eval và ghi file predictions_eval.csv
pred_eval_path = os.path.join(OUT_DIR, "predictions_eval.csv")
final_eval(cfg_final, res_final, data, pred_eval_path)
print(f"Đã lưu dự đoán eval vào: {pred_eval_path}")

# 3. Chạy scripts/evaluate.py để chấm điểm chính thức
eval_result_json = os.path.join(OUT_DIR, "eval_result.json")
eval_cmd = [
    sys.executable,
    os.path.join(REPO_ROOT, "scripts", "evaluate.py"),
    "--pred", pred_eval_path,
    "--data", os.path.join(REPO_ROOT, "data", "covtype.csv.gz"),
    "--meta", os.path.join(REPO_ROOT, "data", "split_metadata.csv"),
    "--out", eval_result_json,
]

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
res_eval_cmd = subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, capture_output=True, text=True)
print(res_eval_cmd.stdout)
if res_eval_cmd.returncode != 0:
    print("LỖI CHẤM EVAL:", res_eval_cmd.stderr)

# Đọc kết quả eval chính thức
with open(eval_result_json, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

final_eval_acc = eval_data["accuracy"]
final_eval_f1 = eval_data["macro_f1"]
print(f"\n=> KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP EVAL:")
print(f"   Accuracy: {final_eval_acc:.4f}")
print(f"   Macro-F1: {final_eval_f1:.4f} (Chỉ số chính)")

# 4. Dự đoán eval cho Baseline để đối chiếu cải thiện
pred_base_path = os.path.join(OUT_DIR, "predictions_base.csv")
final_eval(baseline_results[0]["cfg"], baseline_results[0], data, pred_base_path)
eval_base_json = os.path.join(OUT_DIR, "eval_base_result.json")
eval_base_cmd = [
    sys.executable,
    os.path.join(REPO_ROOT, "scripts", "evaluate.py"),
    "--pred", pred_base_path,
    "--data", os.path.join(REPO_ROOT, "data", "covtype.csv.gz"),
    "--meta", os.path.join(REPO_ROOT, "data", "split_metadata.csv"),
    "--out", eval_base_json,
]
subprocess.run(eval_base_cmd, cwd=REPO_ROOT, env=env, check=True)
with open(eval_base_json, "r", encoding="utf-8") as f:
    base_eval_data = json.load(f)
base_eval_acc = base_eval_data["accuracy"]
base_eval_f1 = base_eval_data["macro_f1"]
print(f"Baseline trên Eval: Acc = {base_eval_acc:.4f} | Macro-F1 = {base_eval_f1:.4f}")
print(f"Mức độ cải thiện của Final Model so với Baseline: {final_eval_f1 - base_eval_f1:+.4f}")


In [ ]:
# Phân tích lỗi theo từng lớp (Per-class Error Analysis)
df_classes = pd.DataFrame(eval_data["per_class"])
df_classes["cls_name"] = [
    "0: Spruce/Fir",
    "1: Lodgepole Pine",
    "2: Ponderosa Pine",
    "3: Cottonwood/Willow",
    "4: Aspen",
    "5: Douglas-fir",
    "6: Krummholz"
]
print("BẢNG ĐÁNH GIÁ THEO LỚP TRÊN EVAL:")
print(df_classes[["cls_name", "support", "precision", "recall", "f1"]].to_string(index=False))

# Vẽ biểu đồ ma trận nhầm lẫn
cm = np.array(eval_data["confusion_matrix"])
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.title("Ma trận nhầm lẫn trên tập Eval (Final Model)")
plt.colorbar()
tick_marks = np.arange(7)
plt.xticks(tick_marks, [f"Lớp {i}" for i in range(7)])
plt.yticks(tick_marks, [f"Lớp {i}" for i in range(7)])
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")

for i in range(7):
    for j in range(7):
        plt.text(j, i, format(cm[i, j], "d"),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black",
                 fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figures", "confusion_matrix_eval.png"), dpi=150)
plt.show()


In [ ]:
# Xuất toàn bộ kết quả vào experiments.xlsx
results_all = load_results(os.path.join(OUT_DIR, "results"))
print(f"Đã đọc {len(results_all)} kết quả thí nghiệm từ {OUT_DIR}/results.")

rows = []
for r in results_all:
    exp_id = r["cfg"]["exp_id"]
    if exp_id == "base-s1":
        r_eval = {"accuracy": base_eval_acc, "macro_f1": base_eval_f1}
    elif exp_id == "final-best":
        r_eval = {"accuracy": final_eval_acc, "macro_f1": final_eval_f1}
    else:
        r_eval = None
    rows.append(to_row(r, eval_scores=r_eval))

template_xlsx = os.path.join(REPO_ROOT, "templates", "experiment_table_template.xlsx")
out_xlsx = os.path.join(OUT_DIR, "experiments.xlsx")

write_xlsx(rows, template_path=template_xlsx, out_path=out_xlsx)
print(f"Đã cập nhật thành công {out_xlsx}!")


## Tổng kết & Kiểm tra Danh mục Nộp bài
- [x] Model tự định nghĩa, đúng shape và đúng **47 879 tham số** cho `M-base`.
- [x] Loss bước 0 $\approx 1.946$ và quá khớp hoàn hảo 20 mẫu.
- [x] Phủ đủ **7 chủ đề thí nghiệm** với đầy đủ ảnh riêng `figures/<exp_id>.png` và biểu đồ so sánh `figures/compare_<nhóm>.png`.
- [x] Bảng `experiments.xlsx` đầy đủ dữ liệu, bảo toàn công thức tự tính.
- [x] File `predictions_eval.csv` hợp lệ và được chấm chính thức ra `eval_result.json`.
- [x] Đã thực hiện phân tích lỗi chi tiết theo từng lớp và ma trận nhầm lẫn.


In [ ]:
# ===== ĐÓNG GÓI THƯ MỤC NỘP BÀI TỰ ĐỘNG =====
import shutil
zip_base = OUT_DIR.rstrip('/\\')
zip_file = f'{zip_base}.zip'
print(f'Đang đóng gói thư mục {OUT_DIR} thành {zip_file}...')
# Nén toàn bộ thư mục nộp bài
shutil.make_archive(zip_base, 'zip', root_dir=os.path.dirname(os.path.abspath(OUT_DIR)) if os.path.dirname(OUT_DIR) else '.', base_dir=os.path.basename(os.path.abspath(OUT_DIR)))
print(f'==> Đã tạo thành công file nén nộp bài: {zip_file}')

if IN_COLAB:
    try:
        from google.colab import files
        print('Tự động tải file zip về máy tính...')
        files.download(zip_file)
    except Exception as e:
        print('Không thể tự động tải file (bạn có thể tải thủ công từ thanh bên Colab):', e)
